# Homework 2: Machine Learning for Regression

## Setup

In [3]:
import pandas as pd
import numpy as np

## Q1. Missing values

In [8]:
df = pd.read_csv('data/car_fuel_efficiency_2026.csv')
missing = df.isnull().sum()
missing[missing > 0]

horsepower      877
acceleration    264
dtype: int64

## Q2. Median for horsepower

In [10]:
df['horsepower'].median()

np.float64(254.0)

## Prepare and split the dataset

In [12]:
cols = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year', 'fuel_efficiency_mpg']
df = df[cols]

In [13]:
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

In [14]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values
y_test = df_test['fuel_efficiency_mpg'].values

del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

In [15]:
def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)
    return w_full[0], w_full[1:]

def train_linear_regression_reg(X, y, r=0.001):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)
    return w_full[0], w_full[1:]

def rmse(y, y_pred):
    se = (y - y_pred) ** 2
    return np.sqrt(se.mean())

## Q3. Filling NAs

In [16]:
# Вариант 1: заполнить нулём
X_train = df_train.fillna(0).values
X_val = df_val.fillna(0).values

w0, w = train_linear_regression(X_train, y_train)
y_pred = w0 + X_val.dot(w)
print('With 0:', round(rmse(y_val, y_pred), 3))

# Вариант 2: заполнить средним (только по train!)
mean_hp = df_train['horsepower'].mean()

X_train = df_train.fillna(mean_hp).values
X_val = df_val.fillna(mean_hp).values

w0, w = train_linear_regression(X_train, y_train)
y_pred = w0 + X_val.dot(w)
print('With mean:', round(rmse(y_val, y_pred), 3))

With 0: 2.205
With mean: 2.202


## Q4.Best regularization

In [17]:
X_train = df_train.fillna(0).values
X_val = df_val.fillna(0).values

for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    w0, w = train_linear_regression_reg(X_train, y_train, r=r)
    y_pred = w0 + X_val.dot(w)
    print(r, round(rmse(y_val, y_pred), 4))

0 2.2053
0.01 2.2058
0.1 2.2241
1 2.3492
5 2.4094
10 2.4195
100 2.4292


In [18]:
def split_data(df, seed):
    n = len(df)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test

    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train = df.iloc[idx[:n_train]].reset_index(drop=True)
    df_val = df.iloc[idx[n_train:n_train + n_val]].reset_index(drop=True)
    df_test = df.iloc[idx[n_train + n_val:]].reset_index(drop=True)

    y_train = df_train.pop('fuel_efficiency_mpg').values
    y_val = df_val.pop('fuel_efficiency_mpg').values
    y_test = df_test.pop('fuel_efficiency_mpg').values

    return df_train, df_val, df_test, y_train, y_val, y_test

## Q5.RMSE standard deviation

In [19]:
scores = []

for seed in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]:
    df_train, df_val, df_test, y_train, y_val, y_test = split_data(df, seed)

    X_train = df_train.fillna(0).values
    X_val = df_val.fillna(0).values

    w0, w = train_linear_regression(X_train, y_train)
    y_pred = w0 + X_val.dot(w)

    score = rmse(y_val, y_pred)
    scores.append(score)
    print(seed, round(score, 3))

print('std:', round(np.std(scores), 3))

0 2.239
1 2.202
2 2.163
3 2.204
4 2.202
5 2.246
6 2.27
7 2.191
8 2.217
9 2.207
std: 0.029


## Q6.Evaluation on test

In [20]:
df_train, df_val, df_test, y_train, y_val, y_test = split_data(df, 9)

df_full_train = pd.concat([df_train, df_val]).reset_index(drop=True)
y_full_train = np.concatenate([y_train, y_val])

X_full_train = df_full_train.fillna(0).values
X_test = df_test.fillna(0).values

w0, w = train_linear_regression_reg(X_full_train, y_full_train, r=0.001)
y_pred = w0 + X_test.dot(w)

print('Test RMSE:', round(rmse(y_test, y_pred), 3))

Test RMSE: 2.236
